# 04 · Evaluación MS-GraphRAG
Evalúa Microsoft GraphRAG (local + global search) sobre UltraDomain con métricas RAGAS.

In [1]:
import os
import sys
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()
sys.path.append(os.path.abspath('../..'))
load_dotenv(os.path.join('../..', '.env'))

# GraphRAG necesita la API key como GRAPHRAG_API_KEY
os.environ['GRAPHRAG_API_KEY'] = os.environ.get('GEMINI_API_KEY', '')

## 1. Configuración

In [2]:
DOMINIO      = "cs"
N_LIBROS     = None
MAX_Q        = None
SHUFFLE      = False
WORKSPACE    = "../../ms_graphrag_workspace"
RESULTS_DIR  = "../../results_MSGraphRAG/"

## 2. Cargar datos

In [3]:
from src.ultradomain import cargar_experimento

libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)
print(f"📚 {len(libros)} libro(s) | {len(qas)} preguntas")

📚 Dominio: cs
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   Total Q&A: 100
📚 10 libro(s) | 100 preguntas


## 3. Cargar índice MS-GraphRAG

In [4]:
from src.baselines.msgraphrag_rag import MSGraphRAG

rag = MSGraphRAG(workspace_dir=WORKSPACE)
rag.load()

📂 Cargando índice MS-GraphRAG...
   ✅ Entidades    : 31,159
   ✅ Relaciones   : 42,358
   ✅ Comunidades  : 4,612
   ✅ Reports      : 4,603
   ✅ Text units   : 1,867


## 4a. Evaluar LOCAL search

In [5]:
from src.evaluation.experiment import run_experiment

result_local = await run_experiment(
    rag_type="msgraphrag_local",
    rag_object=rag,
    dominio=DOMINIO,
    n_libros=N_LIBROS,
    max_questions=MAX_Q,
    shuffle=SHUFFLE,
    save_path=RESULTS_DIR,
)

/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/instructor/providers/gemini/client.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


📚 Dominio: cs
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   Total Q&A: 100

🔍 Evaluando MSGRAPHRAG_LOCAL | cs | 100 preguntas
────────────────────────────────────────────────────────────
  [1/100] How does Spark Streaming enable real-time data processing?...
  [2/100] What does the book suggest about the use of histograms in data anal

## 4b. Evaluar GLOBAL search

In [ ]:
result_global = await run_experiment(
    rag_type="msgraphrag_global",
    rag_object=rag,
    dominio=DOMINIO,
    n_libros=N_LIBROS,
    max_questions=MAX_Q,
    shuffle=SHUFFLE,
    save_path=RESULTS_DIR,
)

## 5. Comparativa local vs global

In [ ]:
print("\n📊 COMPARATIVA LOCAL vs GLOBAL")
print(f"{'Métrica':<25} {'Local':>10} {'Global':>10}")
print("-" * 47)
for metric in ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']:
    local_score  = result_local.ragas_scores.get(metric, float('nan'))
    global_score = result_global.ragas_scores.get(metric, float('nan'))
    print(f"{metric:<25} {local_score:>10.4f} {global_score:>10.4f}")